In [ ]:
!pip install colorama -q

In [ ]:
#!/usr/bin/env python3
import json
import sys
import time
import argparse
import shutil
import urllib.request
import urllib.error
from typing import Optional
from colorama import Fore, Style, init

init(autoreset=True)

FIELDS = "status,message,continent,continentCode,country,countryCode,region,regionName,city,district,zip,lat,lon,timezone,offset,currency,isp,org,as,asname,reverse,mobile,proxy,hosting,query"
API_URL = "http://ip-api.com/json/{ip}?fields=" + FIELDS
BATCH_URL = "http://ip-api.com/batch?fields=" + FIELDS

ascii_banner = r"""
 _   _ _ _   _                 _         ___________   _                     _
| | | | | | (_)               | |       |_   _| ___ \ | |                   | |
| | | | | |_ _ _ __ ___   __ _| |_ ___    | | | |_/ / | |     ___   ___ __ _| |_ ___  _ __
| | | | | __| | '_ ` _ \ / _` | __/ _ \   | | |  __/  | |    / _ \ / __/ _` | __/ _ \| '__|
| |_| | | |_| | | | | | | (_| | ||  __/  _| |_| |     | |___| (_) | (_| (_| | || (_) | |
 \___/|_|\__|_|_| |_| |_|\__,_|\__\___|  \___/\_|     \_____/\___/ \___\__,_|\__\___/|_|
"""

def get_terminal_width():
    try:
        return shutil.get_terminal_size().columns
    except:
        return 80

def center_text(text, width):
    return '\n'.join(line.center(width) for line in text.splitlines())

def format_result(data: dict, show_map_link: bool = True) -> str:
    if data.get("status") == "fail":
        return f"  [ERROR] {data.get('message', 'Unknown error')} for IP: {data.get('query', 'N/A')}\n"

    lines = []
    lines.append(f"\n{'='*55}")
    lines.append(f"  IP Address   : {data.get('query', 'N/A')}")
    lines.append(f"{'='*55}")
    lines.append("  [LOCATION]")
    lines.append(f"  Country      : {data.get('country', 'N/A')} ({data.get('countryCode', 'N/A')})")
    lines.append(f"  Continent    : {data.get('continent', 'N/A')} ({data.get('continentCode', 'N/A')})")
    lines.append(f"  Region       : {data.get('regionName', 'N/A')} ({data.get('region', 'N/A')})")
    lines.append(f"  City         : {data.get('city', 'N/A')}")
    lines.append(f"  District     : {data.get('district') or 'N/A'}")
    lines.append(f"  ZIP Code     : {data.get('zip', 'N/A')}")
    lines.append(f"  Coordinates  : {data.get('lat', 'N/A')}, {data.get('lon', 'N/A')}")
    if show_map_link and data.get('lat') and data.get('lon'):
        lines.append(f"  Map Link     : https://maps.google.com/?q={data['lat']},{data['lon']}")
    lines.append("\n  [TIMEZONE & CURRENCY]")
    lines.append(f"  Timezone     : {data.get('timezone', 'N/A')}")
    offset = data.get('offset', 0)
    sign = '+' if offset >= 0 else ''
    lines.append(f"  UTC Offset   : {sign}{offset//3600}h")
    lines.append(f"  Currency     : {data.get('currency', 'N/A')}")
    lines.append("\n  [ISP / NETWORK]")
    lines.append(f"  ISP          : {data.get('isp', 'N/A')}")
    lines.append(f"  Organization : {data.get('org', 'N/A')}")
    lines.append(f"  AS Number    : {data.get('as', 'N/A')}")
    lines.append(f"  AS Name      : {data.get('asname', 'N/A')}")
    lines.append(f"  Reverse DNS  : {data.get('reverse') or 'N/A'}")
    lines.append("\n  [PROXY / VPN DETECTION]")
    lines.append(f"  Mobile       : {'Yes' if data.get('mobile') else 'No'}")
    lines.append(f"  Proxy/VPN    : {'Yes ⚠' if data.get('proxy') else 'No'}")
    lines.append(f"  Hosting/DC   : {'Yes ⚠' if data.get('hosting') else 'No'}")
    lines.append(f"{'='*55}\n")
    return "\n".join(lines)

def lookup_single(ip: str) -> Optional[dict]:
    url = API_URL.format(ip=ip if ip else "")
    try:
        with urllib.request.urlopen(url, timeout=10) as response:
            return json.loads(response.read().decode())
    except urllib.error.URLError as e:
        print(f"[ERROR] Network error: {e}")
        return None
    except json.JSONDecodeError:
        print("[ERROR] Failed to parse API response.")
        return None

def lookup_batch(ips: list) -> list:
    results = []
    for i in range(0, len(ips), 100):
        chunk = ips[i:i+100]
        payload = json.dumps([{"query": ip} for ip in chunk]).encode()
        req = urllib.request.Request(BATCH_URL, data=payload, headers={"Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(req, timeout=15) as response:
                results.extend(json.loads(response.read().decode()))
        except Exception as e:
            print(f"[ERROR] Batch request failed: {e}")
        if i + 100 < len(ips):
            time.sleep(1)
    return results

def lookup_from_file(filepath: str):
    try:
        with open(filepath, "r") as f:
            ips = [line.strip() for line in f if line.strip() and not line.startswith("#")]
    except FileNotFoundError:
        print(f"[ERROR] File not found: {filepath}")
        sys.exit(1)
    if not ips:
        print("[ERROR] No IPs found in file.")
        sys.exit(1)
    print(f"\n[*] Looking up {len(ips)} IP(s) from '{filepath}'...")
    results = lookup_batch(ips)
    for data in results:
        print(format_result(data))
    proxies = [d for d in results if d.get("proxy")]
    hosting = [d for d in results if d.get("hosting")]
    print(f"[SUMMARY] Total: {len(results)} | Proxy/VPN: {len(proxies)} | Hosting/DC: {len(hosting)}")

In [ ]:
width = get_terminal_width()
print(Fore.RED + center_text(ascii_banner, width))
print(Fore.RED + center_text("created by dzuma youtube.com/@dzumq", width) + Style.RESET_ALL + "\n")

ip = input("Enter IP address to geo-locate (or press Enter for your own IP): ").strip()
data = lookup_single(ip)
if data:
    print(format_result(data))

                                                                                
  _   _ _ _   _                 _         ___________   _                     _ 
| | | | | | (_)               | |       |_   _| ___ \ | |                   | | 
| | | | | |_ _ _ __ ___   __ _| |_ ___    | | | |_/ / | |     ___   ___ __ _| |_ ___  _ __
| | | | | __| | '_ ` _ \ / _` | __/ _ \   | | |  __/  | |    / _ \ / __/ _` | __/ _ \| '__|
| |_| | | |_| | | | | | | (_| | ||  __/  _| |_| |     | |___| (_) | (_| (_| | || (_) | |
 \___/|_|\__|_|_| |_| |_|\__,_|\__\___|  \___/\_|     \_____/\___/ \___\__,_|\__\___/|_|
                      created by dzuma youtube.com/@dzumq                       

Enter IP address to geo-locate (or press Enter for your own IP): 154.57.223.229

  IP Address   : 154.57.223.229
  [LOCATION]
  Country      : Pakistan (PK)
  Continent    : Asia (AS)
  Region       : Punjab (PB)
  City         : Lahore
  District     : N/A
  ZIP Code     : 54030
  Coordinates  : 31.558, 74.3587
